# Step 2: Building the Custom Simulation Environment

In this notebook, we will build a custom simulation environment for our taxi dispatching problem. This environment will be the core of our reinforcement learning project. It will be modeled after the `gymnasium` API to ensure compatibility with standard RL algorithms.

Our environment will have the following components:
1.  **State Space**: Represents the current state of our system, including taxi locations and the time of day.
2.  **Action Space**: The set of possible decisions our agent can make (i.e., which taxi to dispatch).
3.  **Reset Method**: To start a new episode.
4.  **Step Method**: To advance the simulation by one time step, applying an action and receiving a reward.

In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import random

This notebook contains the initial scaffolding for a custom environment, `TaxiDispatchEnv`.

Here is a breakdown of the initial implementation:

- `__init__(self, num_taxis, data_path)`: The constructor initializes the environment. It takes the number of taxis in our fleet and the path to our cleaned Manhattan dataset as input. It also defines the `action_space` (which taxi to dispatch) and a preliminary `observation_space` (the locations of all taxis).
- `reset(self)`: This method prepares the environment for a new episode. It randomly places the taxis in different zones and samples a random trip request from our dataset.
- `step(self, action)`: This is the core of the environment. It takes an action (the index of the taxi to dispatch), calculates a reward, and determines if the episode is over. For now, I've implemented a very simple reward function: the negative distance between the chosen taxi and the pickup location. This is a placeholder, and I will need to have a serious argument about whether this is the right reward for our business objective.
- __Testing Block__: The notebook ends with a few lines of code to instantiate and test the environment, ensuring the basic mechanics are working.

In [3]:
class TaxiDispatchEnv(gym.Env):
    def __init__(self, num_taxis, data_path):
        super(TaxiDispatchEnv, self).__init__()

        self.num_taxis = num_taxis
        self.df = pd.read_parquet(data_path)

        # Define spaces
        self.action_space = spaces.Discrete(num_taxis)
        # For simplicity, state is just the locations of the taxis
        # A more complex state could include time of day, etc.
        self.observation_space = spaces.MultiDiscrete([len(self.df['PULocationID'].unique())] * num_taxis)

        self.reset()

    def reset(self):
        # Initialize taxi locations randomly
        self.taxi_locations = np.random.choice(self.df['PULocationID'].unique(), self.num_taxis)
        
        # Get a random trip request
        self.current_trip = self.df.sample(1).iloc[0]
        
        return self.taxi_locations, {}

    def step(self, action):
        # 'action' is the index of the taxi to dispatch
        chosen_taxi_location = self.taxi_locations[action]
        pickup_location = self.current_trip['PULocationID']
        
        # For now, a simple reward: -1 for every unit of distance between taxi and pickup
        # This is a placeholder. A better reward would use real travel times.
        reward = -abs(chosen_taxi_location - pickup_location)
        
        # For simplicity, the episode ends after one step (one dispatch decision)
        terminated = True
        
        # Get a new trip for the next state
        self.current_trip = self.df.sample(1).iloc[0]
        
        return self.taxi_locations, reward, terminated, False, {}

    def render(self, mode='human'):
        pass

### Testing the Environment

Let's instantiate the environment and test its basic functionality.

In [4]:
NUM_TAXIS = 5
DATA_PATH = 'data/yellow_tripdata_2025-01_manhattan.parquet'

env = TaxiDispatchEnv(num_taxis=NUM_TAXIS, data_path=DATA_PATH)

# Reset the environment and get the initial state
initial_state, _ = env.reset()
print(f"Initial taxi locations: {initial_state}")

# Take a random action
random_action = env.action_space.sample()
print(f"Dispatching taxi index: {random_action}")

# Perform a step
next_state, reward, terminated, _, _ = env.step(random_action)

print(f"Next state (taxi locations): {next_state}")
print(f"Reward: {reward}")
print(f"Terminated: {terminated}")

Initial taxi locations: [262 144  88 125 153]
Dispatching taxi index: 1
Next state (taxi locations): [262 144  88 125 153]
Reward: -26
Terminated: True


### V2: A Richer Environment

Here is the `TaxiDispatchEnvV2` class. It includes the richer state space and state evolution logic we discussed. It is a significant improvement over the first version.

In [5]:
class TaxiDispatchEnvV2(gym.Env):
    """
    A more sophisticated taxi dispatch environment.

    State: A dictionary with 'taxis', 'request', and 'time'.
    - 'taxis': An array of current taxi LocationIDs.
    - 'request': A dictionary with 'origin' and 'destination' LocationIDs.
    - 'time': A dictionary with 'day_of_week' and 'hour_of_day'.

    Action: An integer representing the index of the taxi to dispatch.
    """
    def __init__(self, num_taxis, data_path):
        super(TaxiDispatchEnvV2, self).__init__()

        self.num_taxis = num_taxis
        
        # Load and preprocess data
        self.df = pd.read_parquet(data_path)
        self.df['tpep_pickup_datetime'] = pd.to_datetime(self.df['tpep_pickup_datetime'])
        self.df['day_of_week'] = self.df['tpep_pickup_datetime'].dt.dayofweek
        self.df['hour_of_day'] = self.df['tpep_pickup_datetime'].dt.hour
        
        # Get unique locations for defining the space
        self.unique_locations = sorted(pd.concat([self.df['PULocationID'], self.df['DOLocationID']]).unique())
        self.location_map = {loc: i for i, loc in enumerate(self.unique_locations)}
        self.inverse_location_map = {i: loc for i, loc in enumerate(self.unique_locations)}
        num_locations = len(self.unique_locations)

        # Define spaces
        self.action_space = spaces.Discrete(num_taxis)
        
        self.observation_space = spaces.Dict({
            'taxis': spaces.MultiDiscrete([num_locations] * num_taxis),
            'request': spaces.Dict({
                'origin': spaces.Discrete(num_locations),
                'destination': spaces.Discrete(num_locations)
            }),
            'time': spaces.Dict({
                'day_of_week': spaces.Discrete(7), # 0=Monday, 6=Sunday
                'hour_of_day': spaces.Discrete(24)
            })
        })

    def _get_observation(self):
        # Map real LocationIDs to the discrete space indices
        taxi_indices = [self.location_map[loc] for loc in self.taxi_locations]
        origin_index = self.location_map[self.current_trip['PULocationID']]
        dest_index = self.location_map[self.current_trip['DOLocationID']]

        return {
            'taxis': np.array(taxi_indices),
            'request': {
                'origin': origin_index,
                'destination': dest_index
            },
            'time': {
                'day_of_week': self.current_trip['day_of_week'],
                'hour_of_day': self.current_trip['hour_of_day']
            }
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Initialize taxi locations randomly to real LocationIDs
        self.taxi_locations = np.random.choice(self.unique_locations, self.num_taxis)
        
        # Get a random trip request
        self.current_trip = self.df.sample(1, random_state=self.np_random).iloc[0]
        
        return self._get_observation(), {}

    def step(self, action):
        # 'action' is the index of the taxi to dispatch
        chosen_taxi_location_id = self.taxi_locations[action]
        pickup_location_id = self.current_trip['PULocationID']
        
        # Reward is the negative of the absolute difference in LocationIDs (a proxy for distance)
        reward = -abs(chosen_taxi_location_id - pickup_location_id)
        
        # Update the dispatched taxi's location to the dropoff location of the completed trip
        self.taxi_locations[action] = self.current_trip['DOLocationID']
        
        # Get a new trip for the next state
        self.current_trip = self.df.sample(1, random_state=self.np_random).iloc[0]
        
        terminated = True # Each step is one decision episode
        
        return self._get_observation(), reward, terminated, False, {}

    def render(self, mode='human'):
        pass

### Testing the V2 Environment

Now, let's test the new environment. Notice how the state is now a dictionary, providing much more context to the agent.

In [11]:
# --- Test Cell for V2 ---
env_v2 = TaxiDispatchEnvV2(num_taxis=NUM_TAXIS, data_path=DATA_PATH)

# Reset the environment and get the initial state
initial_state, _ = env_v2.reset()
print("--- Initial State (V2) ---")
print(initial_state)

# Take a random action
random_action = env_v2.action_space.sample()
print(f"\nDispatching taxi index: {random_action}")

# Perform a step
next_state, reward, terminated, _, _ = env_v2.step(random_action)

print("\n--- Next State (V2) ---")
print(next_state)
print(f"\nReward: {reward}")
print(f"Terminated: {terminated}")

--- Initial State (V2) ---
{'taxis': array([14, 42, 52, 47,  0]), 'request': {'origin': 38, 'destination': 57}, 'time': {'day_of_week': np.int32(3), 'hour_of_day': np.int32(21)}}

Dispatching taxi index: 4

--- Next State (V2) ---
{'taxis': array([14, 42, 52, 47, 57]), 'request': {'origin': 54, 'destination': 51}, 'time': {'day_of_week': np.int32(4), 'hour_of_day': np.int32(22)}}

Reward: -157
Terminated: True


In [12]:
class TaxiDispatchEnvV3(gym.Env):
    """
    The definitive taxi dispatch environment.

    This version incorporates a pre-computed travel matrix for realistic, time-aware
    cost calculations and uses a profit-maximizing reward function.

    State: A dictionary with 'taxis', 'request', and 'time'.
    - 'taxis': An array of current taxi LocationIDs.
    - 'request': A dictionary with 'origin' and 'destination' LocationIDs.
    - 'time': A dictionary with 'day_of_week' and 'hour_of_day'.

    Action: An integer representing the index of the taxi to dispatch.
    """
    def __init__(self, num_taxis, data_path, matrix_path, cost_per_mile=1.5):
        super(TaxiDispatchEnvV3, self).__init__()

        self.num_taxis = num_taxis
        self.cost_per_mile = cost_per_mile
        
        # Load trip data
        self.df = pd.read_parquet(data_path)
        self.df['tpep_pickup_datetime'] = pd.to_datetime(self.df['tpep_pickup_datetime'])
        self.df['day_of_week'] = self.df['tpep_pickup_datetime'].dt.dayofweek
        self.df['hour_of_day'] = self.df['tpep_pickup_datetime'].dt.hour
        
        # Load and index the travel matrix for fast lookups
        travel_matrix_df = pd.read_parquet(matrix_path)
        self.travel_matrix = travel_matrix_df.set_index(['PULocationID', 'DOLocationID', 'day_of_week', 'hour_of_day'])

        # Get unique locations for defining the space
        self.unique_locations = sorted(pd.concat([self.df['PULocationID'], self.df['DOLocationID']]).unique())
        self.location_map = {loc: i for i, loc in enumerate(self.unique_locations)}
        num_locations = len(self.unique_locations)

        # Define spaces
        self.action_space = spaces.Discrete(num_taxis)
        
        self.observation_space = spaces.Dict({
            'taxis': spaces.MultiDiscrete([num_locations] * num_taxis),
            'request': spaces.Dict({
                'origin': spaces.Discrete(num_locations),
                'destination': spaces.Discrete(num_locations)
            }),
            'time': spaces.Dict({
                'day_of_week': spaces.Discrete(7),
                'hour_of_day': spaces.Discrete(24)
            })
        })

    def _get_observation(self):
        taxi_indices = [self.location_map[loc] for loc in self.taxi_locations]
        origin_index = self.location_map[self.current_trip['PULocationID']]
        dest_index = self.location_map[self.current_trip['DOLocationID']]

        return {
            'taxis': np.array(taxi_indices),
            'request': {'origin': origin_index, 'destination': dest_index},
            'time': {'day_of_week': self.current_trip['day_of_week'], 'hour_of_day': self.current_trip['hour_of_day']}
        }

    def _get_travel_cost(self, origin_id, dest_id, day, hour):
        """Looks up the travel distance from the pre-computed matrix."""
        try:
            # Look up the specific time
            cost = self.travel_matrix.loc[(origin_id, dest_id, day, hour)]
            return cost['mean_distance']
        except KeyError:
            # Fallback: if no data for this specific hour, try to get the average for the day
            try:
                cost = self.travel_matrix.loc[(origin_id, dest_id, day)].mean()
                return cost['mean_distance']
            except KeyError:
                # Fallback: if no data for this route at all, return a high penalty distance
                return 10.0 # High penalty for unknown routes

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.taxi_locations = np.random.choice(self.unique_locations, self.num_taxis)
        self.current_trip = self.df.sample(1, random_state=self.np_random).iloc[0]
        return self._get_observation(), {}

    def step(self, action):
        chosen_taxi_location_id = self.taxi_locations[action]
        
        # Get trip details from the current request
        pickup_location_id = self.current_trip['PULocationID']
        fare = self.current_trip['fare_amount']
        actual_trip_distance = self.current_trip['trip_distance']
        day = self.current_trip['day_of_week']
        hour = self.current_trip['hour_of_day']

        # Calculate deadhead distance using our travel matrix
        deadhead_distance = self._get_travel_cost(chosen_taxi_location_id, pickup_location_id, day, hour)
        
        # Calculate total cost
        total_distance = deadhead_distance + actual_trip_distance
        total_cost = total_distance * self.cost_per_mile
        
        # Calculate profit-based reward
        reward = fare - total_cost
        
        # Update the dispatched taxi's location to the dropoff of the completed trip
        self.taxi_locations[action] = self.current_trip['DOLocationID']
        
        # Get a new trip for the next state
        self.current_trip = self.df.sample(1, random_state=self.np_random).iloc[0]
        
        terminated = True
        
        return self._get_observation(), reward, terminated, False, {}

    def render(self, mode='human'):
        pass

In [13]:
# --- Test Cell for V3 ---
MATRIX_PATH = 'data/travel_matrix.parquet'
env_v3 = TaxiDispatchEnvV3(num_taxis=NUM_TAXIS, data_path=DATA_PATH, matrix_path=MATRIX_PATH)

# Reset the environment
initial_state, _ = env_v3.reset()
print("--- Initial State (V3) ---")
print(initial_state)

# Take a random action
random_action = env_v3.action_space.sample()
print(f"\nDispatching taxi index: {random_action}")

# Perform a step
next_state, reward, terminated, _, _ = env_v3.step(random_action)

print("\n--- Next State (V3) ---")
print(next_state)
print(f"\nReward (Profit): {reward:.2f}")
print(f"Terminated: {terminated}")

--- Initial State (V3) ---
{'taxis': array([31, 58, 47, 48, 60]), 'request': {'origin': 6, 'destination': 57}, 'time': {'day_of_week': np.int32(1), 'hour_of_day': np.int32(10)}}

Dispatching taxi index: 0

--- Next State (V3) ---
{'taxis': array([57, 58, 47, 48, 60]), 'request': {'origin': 10, 'destination': 16}, 'time': {'day_of_week': np.int32(1), 'hour_of_day': np.int32(19)}}

Reward (Profit): 7.14
Terminated: True
